# Business Capability Tree — Step-by-Step Test Notebook

This notebook builds and refines the business capability hierarchy:
1. **Setup** — Configuration, connections
2. **Build** — Parse EP Business Map workbook into capability tree
3. **Inspect** — Review tree structure
4. **Upload** — Push to FalkorDB (shared graph with org hierarchy)
5. **Verify** — Check FalkorDB data
6. **Refine** — LLM-based description enrichment
7. **Semantic search** — Test refined embeddings
8. **Export** — Save refinements to JSON

---
## Step 1: Configuration

In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '.')
load_dotenv()

# Capability source (EP Business Map workbook)
CAPABILITY_SOURCE = os.getenv(
    "CAPABILITY_SOURCE",
    "./data/Capability/EP_Business_Map_5.0.xlsx"
)
CAPABILITY_SHEET = os.getenv("CAPABILITY_SHEET", "Business Map")

# FalkorDB — shared graph with org hierarchy
FALKORDB_URL = os.getenv("FALKORDB_URL", "")
CAPABILITY_GRAPH_NAME = os.getenv("CAPABILITY_GRAPH_NAME", "org_hierarchy")

# Vector store (for embeddings)
VECTOR_STORE_TYPE = os.getenv("VECTOR_STORE_TYPE", "mock")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

# Refinement
REFINEMENT_STRATEGY = "top_down"   # top_down, bottom_up, bidirectional
USE_MOCK_LLM = False
MAX_NODES = None                   # Set to e.g. 5 for quick testing

print("Configuration:")
print(f"  Source:          {CAPABILITY_SOURCE}")
print(f"  Sheet:           {CAPABILITY_SHEET}")
print(f"  Graph name:      {CAPABILITY_GRAPH_NAME}")
print(f"  Strategy:        {REFINEMENT_STRATEGY}")
print(f"  Mock LLM:        {USE_MOCK_LLM}")
print(f"  Max nodes:       {MAX_NODES or 'all'}")
masked = FALKORDB_URL.split("@")[-1] if "@" in FALKORDB_URL else "(not set)"
print(f"  FalkorDB host:   {masked}")

Configuration:
  Source:          ./data/Capability/EP_Business_Map_5.0.xlsx
  Sheet:           Business Map
  Graph name:      org_hierarchy
  Strategy:        top_down
  Mock LLM:        False
  Max nodes:       all
  FalkorDB host:   r-6jissuruar.instance-zyadijt4h.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:55831


---
## Step 2: Build Capability Tree from EP Business Map

In [2]:
# =============================================================================
# BUILD CAPABILITY TREE
# =============================================================================

from capability_graph import create_capability_builder

builder = create_capability_builder(
    data_path=CAPABILITY_SOURCE,
    sheet_name=CAPABILITY_SHEET,
    falkordb_url=FALKORDB_URL,
    falkordb_graph_name=CAPABILITY_GRAPH_NAME,
    verbose=True
)

graph = builder.graph
print(f"\n✓ Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"  Categories (L0): {len(builder.categories)}")

✓ FalkorDB connected: org_hierarchy

BUILDING CAPABILITY GRAPH
Source: ./data/Capability/EP_Business_Map_5.0.xlsx
Rows: 63  |  Columns: ['Category', 'Category Description', 'Business Area', 'Business Area Description', 'Sub-Business Area', 'Sub-Business Area Definition', 'Old Sub-Business Area Definition']
Categories (L0):         3
Business Areas (L1):     15
Sub-Business Areas (L2): 63
Total nodes:             81
Total edges:             78

✓ Graph: 81 nodes, 78 edges
  Categories (L0): 3


---
## Step 3: Inspect Capability Tree

In [3]:
# =============================================================================
# PRINT TREE
# =============================================================================

builder.print_tree()

[ROOT] EP Business Capability Map
  ├── [L0] CORE
  │   ├── [L1] Political Engagement and Relationship Managem
  │   │   ├── [L2] Engagement with Citizens
  │   │   ├── [L2] Strategic Stakeholder relationship management
  │   │   └── [L2] Enlargement of EU
  │   ├── [L1] Legislative, Budgetary and Non-Legislative Po
  │   │   ├── [L2] International agreements
  │   │   ├── [L2] Non-legislative procedures
  │   │   ├── [L2] Relations with National Parliaments
  │   │   ├── [L2] EU Legislation examination, adoption and cont
  │   │   └── [L2] EU Budget adoption and control
  │   └── [L1] Empowering parliamentary work
  │       ├── [L2] Interpretation
  │       ├── [L2] Legal Services
  │       ├── [L2] Plenary sittings organisation
  │       ├── [L2] Committees' parliamentary activities
  │       ├── [L2] Elections and Establishment of Parliamentary 
  │       ├── [L2] Intercultural linguistic mediation
  │       ├── [L2] Policy advice, analysis and research
  │       └── [L2] Planning &

In [4]:
# =============================================================================
# INSPECT BY LEVEL
# =============================================================================

print("=" * 70)
print("CAPABILITIES BY LEVEL")
print("=" * 70)

for level in range(3):
    nodes = [n for n, d in graph.nodes(data=True) if d.get("level") == level]
    print(f"\nLevel {level}: {len(nodes)} nodes")
    for nid in sorted(nodes)[:10]:
        data = graph.nodes[nid]
        name = data.get("name", "")[:50]
        desc = data.get("description", "")[:60]
        print(f"  [{nid:25s}] {name}")
        if desc:
            print(f"    {desc}...")
    if len(nodes) > 10:
        print(f"  ... +{len(nodes) - 10} more")

CAPABILITIES BY LEVEL

Level 0: 3 nodes
  [L1_CORE_c5d5df           ] CORE
    The activities that are core and fundamental to the purpose ...
  [L1_STEERING_4c714c       ] STEERING
    The planning, measuring, monitoring and governance activitie...
  [L1_SUPPORTING_59bd41     ] SUPPORTING
    The provision and management of internal EP services, resour...

Level 1: 15 nodes
  [L2_1870dbc45b71          ] Human Capital
    Ability to strategically manage the organisation's human cap...
  [L2_1a356b457803          ] Information Technology
    This business area refers to anything related to computing t...
  [L2_1b9b73f15046          ] Political Engagement and Relationship Management
    The purpose of this business area together with the Communic...
  [L2_1bb9410bc69b          ] Communication
    Communication is the business area which supports: 1. extern...
  [L2_28570407d618          ] Technical Infrastructure Management
    The Technical Infrastructure Management (TIM) capability is 

---
## Step 4: Upload to FalkorDB

In [5]:
# =============================================================================
# UPLOAD TO FALKORDB
# =============================================================================

if builder._connected:
    count = builder.upload_to_falkordb(clear_existing=True, create_index=True)
    print(f"\n✓ Uploaded {count} capability nodes")
else:
    print("⚠ Not connected to FalkorDB — skipping upload")
    print("  Set FALKORDB_URL in .env to enable")


UPLOADING CAPABILITY GRAPH TO FALKORDB
  Cleared existing capability data (org hierarchy preserved)
✓ Capability root: CAPABILITY_ROOT
  Uploaded 81 capability nodes
  Created 78 edges
  Connected 3 categories to root
✓ Vector index: Capability.description_embedding

✓ Upload complete:
  Nodes:      82
  Edges:      81
  Categories: 3

✓ Uploaded 81 capability nodes


---
## Step 5: Verify FalkorDB

In [6]:
# =============================================================================
# VERIFY FALKORDB
# =============================================================================

if builder._connected:
    stats = builder.get_stats()
    print("FalkorDB Capability Graph:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

    # Check labels
    labels = builder.query(
        "MATCH (n:Capability) "
        "RETURN labels(n) as lbl, count(*) as cnt "
        "ORDER BY cnt DESC"
    )
    print("\nNode labels:")
    for row in labels:
        print(f"  {row.get('lbl')}: {row.get('cnt')}")

    # Sample nodes
    sample = builder.query(
        "MATCH (n:Category) "
        "RETURN n.node_id as id, n.name as name "
        "ORDER BY n.name LIMIT 10"
    )
    print("\nCategories:")
    for row in sample:
        print(f"  [{row.get('id')}] {row.get('name')}")
else:
    print("⚠ Not connected to FalkorDB")

FalkorDB Capability Graph:
  connected: True
  nodes: 82
  edges: 81
  categories: 3

Node labels:
  ['Capability', 'SubBusinessArea']: 63
  ['Capability', 'BusinessArea']: 15
  ['Capability', 'Category']: 3
  ['Capability', 'CapabilityRoot']: 1

Categories:
  [L1_CORE_c5d5df] CORE
  [L1_STEERING_4c714c] STEERING
  [L1_SUPPORTING_59bd41] SUPPORTING


---
## Step 6: Refine Capability Descriptions

Use LLM to enrich capability descriptions with context and semantic keywords.

In [7]:
# =============================================================================
# CREATE LLM & VECTOR STORE
# =============================================================================

from llm import create_llm
from storage_base import StorageConfig
from storage_impl import create_vector_store

llm = create_llm(use_mock=USE_MOCK_LLM)

storage_config = StorageConfig(embedding_model=EMBEDDING_MODEL)
vector_store = create_vector_store(VECTOR_STORE_TYPE, storage_config)
print(f"✓ Vector store: {type(vector_store).__name__}")

C:\Users\marci\anaconda3\envs\DATAENLIGHT_AI_ARCH_PATTERNS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ LLM: claude-sonnet-4-20250514
✓ VectorStore: Mock (in-memory)
✓ Vector store: MockVectorStore


In [8]:
# =============================================================================
# CREATE REFINEMENT AGENT
# =============================================================================

from capability_refinement import create_capability_refinement_agent

agent = create_capability_refinement_agent(
    capability_builder=builder,
    llm=llm,
    vector_store=vector_store,
    strategy=REFINEMENT_STRATEGY,
    sync_to_falkordb=builder._connected,
    update_embeddings=builder._connected,
    create_vector_index=builder._connected,
    embedding_dimension=384,
    verbose=True,
    max_nodes=MAX_NODES
)

print(f"\n✓ Refinement agent created")
print(f"  Strategy:  {agent.config.strategy}")
print(f"  FalkorDB:  {agent.config.sync_to_falkordb}")


✓ Refinement agent created
  Strategy:  top_down
  FalkorDB:  True


In [9]:
# =============================================================================
# TEST: REFINE A SINGLE CAPABILITY
# =============================================================================

from capability_refinement import get_capability_context

# Pick a leaf capability (L2) for testing
test_cap = None
for nid in sorted(graph.nodes()):
    if graph.nodes[nid].get("level") == 2:
        test_cap = nid
        break

if test_cap:
    data = graph.nodes[test_cap]
    ctx = get_capability_context(graph, test_cap)

    print("=" * 70)
    print(f"TEST: [{test_cap}] {data.get('name', '')}")
    print("=" * 70)
    print(f"  Level: {data.get('level')}")
    print(f"  Description: {data.get('description', '')[:150]}...")
    print(f"  Parent: {ctx['parent_info'][:80]}")

    result = agent.refine_capability(test_cap)
    print(f"\n  Refined: {result['refined_description'][:200]}...")
    print(f"  Keywords: {result['capability_keywords']}")
else:
    print("No L2 capabilities found")

TEST: [L3_03f8c22d98bf] Interpretation
  Level: 2
  Description: Interpretation is a facility governed by the Code of Conduct on multilingualism and offered to EP Members and official bodies to enable them to commun...
  Parent: Empowering parliamentary work

  Refined: The Interpretation capability provides essential multilingual communication services that enable Members of the European Parliament and official bodies to overcome linguistic barriers during parliamen...
  Keywords: interpretation, multilingual communication, oral linguistic support, simultaneous interpretation, consecutive interpretation, language barriers, parliamentary meetings, political meetings, multilingualism, linguistic services, translation services, communication facilitation, language diversity, cross-language communication, interpreter services


In [10]:
# =============================================================================
# RUN FULL REFINEMENT
# =============================================================================

# Clear test refinement first
for n in graph.nodes():
    graph.nodes[n]["refined_description"] = ""
    graph.nodes[n]["capability_keywords"] = ""

summary = agent.run()

print("\n" + "=" * 70)
print("REFINEMENT SUMMARY")
print("=" * 70)
for k, v in summary.items():
    print(f"  {k}: {v}")


CAPABILITY REFINEMENT AGENT
  Strategy:          top_down
  Sync to FalkorDB:  True
  Update embeddings: True
  Vector indexes:    True

══════════════════════════════════════════════════════════════════════
CREATING CAPABILITY VECTOR INDEXES
══════════════════════════════════════════════════════════════════════
✓ Vector index: Capability.refined_embedding

══════════════════════════════════════════════════════════════════════
CAPABILITY TOP-DOWN REFINEMENT (Categories → Leaves)
══════════════════════════════════════════════════════════════════════
Processing 81 capabilities...

[  1/81] L1_STEERING_4c714c   STEERING
           → The STEERING capability represents the foundationa...

[  2/81] L1_CORE_c5d5df       CORE
           → The CORE capability category encompasses the funda...

[  3/81] L1_SUPPORTING_59bd41 SUPPORTING
           → The Supporting capability category encompasses all...

[  4/81] L2_55d9653ce231      Planning
           → The Planning capability encompasses the sy

---
## Step 7: Semantic Search on Refined Capabilities

In [14]:
# =============================================================================
# SEMANTIC SEARCH
# =============================================================================

test_queries = [
    "event management and conference organization",
    "cybersecurity and information protection",
    "software development and application lifecycle",
    "financial planning and budget management",
    "data analytics and business intelligence",
]

print("=" * 70)
print("SEMANTIC SEARCH — CAPABILITY EMBEDDINGS")
print("=" * 70)

if builder._connected:
    for q in test_queries:
        print(f"\nQuery: '{q}'")
        try:
            emb = vector_store.get_embedding(q)
            results = builder.query(
                "CALL db.idx.vector.queryNodes("
                "'Capability', 'refined_embedding', 5, vecf32($emb)"
                ") YIELD node, score "
                "RETURN node.node_id as id, node.name as name, "
                "node.level as level, score "
                "ORDER BY score DESC",
                {"emb": emb}
            )
            for row in results:
                print(f"  L{row.get('level')} [{row.get('id')}] "
                      f"(score: {row.get('score',0):.3f}) {row.get('name')}")
        except Exception as e:
            print(f"  Error: {e}")
else:
    print("⚠ FalkorDB not connected — semantic search requires uploaded embeddings")

SEMANTIC SEARCH — CAPABILITY EMBEDDINGS

Query: 'event management and conference organization'
  L2 [L3_2c14dfe56f21] (score: 0.696) Service Management
  L2 [L3_4f937c99d5f1] (score: 0.692) Selection & Recruitment
  L2 [L3_7c4676d165fd] (score: 0.688) IT Infrastructure Management
  L2 [L3_c11c9ddc324f] (score: 0.643) Buildings
  L2 [L3_1b37abf59083] (score: 0.526) Citizen and stakeholder Communication

Query: 'cybersecurity and information protection'
  L0 [L1_STEERING_4c714c] (score: 0.728) STEERING
  L2 [L3_748e908611ea] (score: 0.694) Conference management
  L2 [L3_f06a8e555282] (score: 0.685) Safety
  L2 [L3_2196e3b6e660] (score: 0.634) Non-legislative procedures
  L1 [L2_28cdcde30b34] (score: 0.625) Facilities and Auxiliary Services

Query: 'software development and application lifecycle'
  L1 [L2_1870dbc45b71] (score: 0.681) Human Capital
  L1 [L2_55d9653ce231] (score: 0.666) Planning
  L0 [L1_CORE_c5d5df] (score: 0.661) CORE
  L2 [L3_bed59401b9f5] (score: 0.658) Strategic Stakeh

---
## Step 8: Export

In [15]:
# =============================================================================
# EXPORT REFINEMENTS
# =============================================================================

agent.export_refinements("./output/capability_refinements.json")
builder.save_local()

✓ Exported: ./output/capability_refinements.json (81 capabilities)
✓ Saved: ./graph_data/capability_graph.json


'./graph_data/capability_graph.json'

---
## Summary

In [16]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("CAPABILITY PIPELINE SUMMARY")
print("=" * 70)

total = graph.number_of_nodes()
refined = sum(1 for n in graph.nodes() if graph.nodes[n].get("refined_description"))
kw = sum(1 for n in graph.nodes() if graph.nodes[n].get("capability_keywords"))

print(f"  Capability source:   {CAPABILITY_SOURCE}")
print(f"  FalkorDB graph:      {CAPABILITY_GRAPH_NAME}")
print(f"  Total capabilities:  {total}")
print(f"  Categories (L0):     {len(builder.categories)}")
l1 = sum(1 for _, d in graph.nodes(data=True) if d.get("level") == 1)
l2 = sum(1 for _, d in graph.nodes(data=True) if d.get("level") == 2)
print(f"  Business Areas (L1): {l1}")
print(f"  Sub-Areas (L2):      {l2}")
print(f"  Refined:             {refined}")
print(f"  With keywords:       {kw}")
print(f"  FalkorDB synced:     {agent._stats.get('synced_to_falkordb', 0)}")
print(f"  Embeddings updated:  {agent._stats.get('embeddings_updated', 0)}")
print("\n✓ Capability pipeline complete")


CAPABILITY PIPELINE SUMMARY
  Capability source:   ./data/Capability/EP_Business_Map_5.0.xlsx
  FalkorDB graph:      org_hierarchy
  Total capabilities:  81
  Categories (L0):     3
  Business Areas (L1): 15
  Sub-Areas (L2):      63
  Refined:             81
  With keywords:       81
  FalkorDB synced:     81
  Embeddings updated:  81

✓ Capability pipeline complete
